# 03 - Chunking estructural del corpus normativo ambiental

Este notebook genera fragmentos estructurados (*chunks*) a partir de los textos extraídos de los PDFs.

Entrada principal:

- `data/metadata/corpus_normativo_ambiental_con_extraccion.csv`
- textos `.txt` ubicados en `data/processed/`

Salida generada:

- `data/chunks/chunks_normativa_v1.jsonl`
- `data/chunks/chunks_normativa_v1.csv`
- reportes en `experiments/resultados/`

El objetivo es dividir los documentos respetando, en la medida de lo posible, la estructura normativa:

- títulos;
- capítulos;
- artículos;
- incisos;
- disposiciones;
- anexos.

Estos chunks serán usados luego para generar embeddings e indexarlos en la base vectorial.


## 1. Importación de librerías

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import re

try:
    import pandas as pd
    print("pandas importado correctamente.")
except ImportError:
    raise ImportError(
        "No se encontró pandas. Instálalo con: pip install pandas"
    )

print("Librerías importadas correctamente.")


## 2. Definición de rutas del proyecto

El notebook puede ejecutarse desde la raíz del repositorio o desde la carpeta `notebooks/`.


In [ ]:
current_dir = Path.cwd()

if current_dir.name == "notebooks":
    ROOT_DIR = current_dir.parent
else:
    ROOT_DIR = current_dir

DATA_DIR = ROOT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
CHUNKS_DIR = DATA_DIR / "chunks"
METADATA_DIR = DATA_DIR / "metadata"
REPORTS_DIR = ROOT_DIR / "experiments" / "resultados"

CSV_EXTRACTION_PATH = METADATA_DIR / "corpus_normativo_ambiental_con_extraccion.csv"
CSV_BASE_PATH = METADATA_DIR / "corpus_normativo_ambiental.csv"

CHUNKS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

if CSV_EXTRACTION_PATH.exists():
    CSV_PATH = CSV_EXTRACTION_PATH
else:
    CSV_PATH = CSV_BASE_PATH

print(f"ROOT_DIR: {ROOT_DIR}")
print(f"CSV_PATH: {CSV_PATH}")
print(f"PROCESSED_DIR: {PROCESSED_DIR}")
print(f"CHUNKS_DIR: {CHUNKS_DIR}")


## 3. Carga del corpus

In [ ]:
df = pd.read_csv(CSV_PATH)

print(f"Documentos registrados: {len(df)}")
df.head()


## 4. Parámetros del chunking

Estos valores pueden ajustarse durante la fase de experimentación.

- `MAX_WORDS`: tamaño máximo recomendado por chunk.
- `OVERLAP_WORDS`: solapamiento usado cuando un bloque normativo es demasiado largo.
- `MIN_WORDS`: mínimo de palabras para conservar un chunk.


In [ ]:
MAX_WORDS = 450
OVERLAP_WORDS = 80
MIN_WORDS = 25

print(f"MAX_WORDS: {MAX_WORDS}")
print(f"OVERLAP_WORDS: {OVERLAP_WORDS}")
print(f"MIN_WORDS: {MIN_WORDS}")


## 5. Expresiones regulares para detectar estructura normativa

Se usan patrones flexibles para documentos peruanos, considerando formas como:

- `TÍTULO I`
- `CAPÍTULO II`
- `Artículo 1º.-`
- `Artículo 1.-`
- `DISPOSICIONES COMPLEMENTARIAS`
- `ANEXO Nº 01`


In [ ]:
PAGE_MARKER_RE = re.compile(r"^=+\s*PÁGINA\s+(\d+)\s*=+$", re.IGNORECASE)

TITLE_RE = re.compile(
    r"^(T[ÍI]TULO\s+(?:PRELIMINAR|[IVXLCDM]+|\d+).*)$",
    re.IGNORECASE
)

CHAPTER_RE = re.compile(
    r"^(CAP[ÍI]TULO\s+(?:[IVXLCDM]+|\d+).*)$",
    re.IGNORECASE
)

ARTICLE_RE = re.compile(
    r"^(Art[íi]culo|Articulo)\s+\d+[A-Za-z0-9]*[°º]?\s*[\.\-–—º]*\s*.*$",
    re.IGNORECASE
)

ANNEX_RE = re.compile(
    r"^(ANEXO\s*(?:N[°º]\s*)?(?:\d+|[IVXLCDM]+)?.*)$",
    re.IGNORECASE
)

DISPOSITION_RE = re.compile(
    r"^((DISPOSICI[ÓO]N|DISPOSICIONES)\s+(COMPLEMENTARIAS|FINALES|TRANSITORIAS|MODIFICATORIAS).*)$",
    re.IGNORECASE
)

LEGAL_HEADER_RE = re.compile(
    r"^(DECRETO SUPREMO|RESOLUCI[ÓO]N MINISTERIAL|LEY|DECRETO LEGISLATIVO|RESOLUCI[ÓO]N DIRECTORAL).*$",
    re.IGNORECASE
)

def normalize_line(line: str) -> str:
    if not isinstance(line, str):
        return ""
    line = line.replace("\t", " ")
    line = re.sub(r" {2,}", " ", line)
    return line.strip()


def normalize_text_for_chunk(text: str) -> str:
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()


def count_words(text: str) -> int:
    return len(re.findall(r"\b\w+\b", text, flags=re.UNICODE))


## 6. Funciones principales de chunking

In [ ]:
def split_large_text(text: str, max_words: int = MAX_WORDS, overlap_words: int = OVERLAP_WORDS):
    """
    Divide un texto largo en subchunks con solapamiento.
    Se usa solo cuando un artículo, anexo o bloque excede el tamaño máximo.
    """
    words = text.split()

    if len(words) <= max_words:
        return [text]

    chunks = []
    start = 0

    while start < len(words):
        end = start + max_words
        chunk_words = words[start:end]
        chunks.append(" ".join(chunk_words))

        if end >= len(words):
            break

        start = max(0, end - overlap_words)

    return chunks


def create_chunk_record(
    row,
    text,
    tipo_chunk,
    seccion,
    titulo_seccion,
    capitulo,
    anexo,
    pagina_inicio,
    pagina_fin,
    orden_chunk,
    suborden=1
):
    """
    Construye un registro de chunk con metadatos documentales.
    """
    chunk_id = f"{row['id_documento']}_CHK-{orden_chunk:04d}"

    if suborden > 1:
        chunk_id = f"{chunk_id}_P{suborden:02d}"

    clean_chunk_text = normalize_text_for_chunk(text)

    return {
        "id_chunk": chunk_id,
        "id_documento": row.get("id_documento", "No determinado"),
        "archivo_pdf": row.get("archivo_pdf", "No determinado"),
        "archivo_txt": row.get("archivo_txt", f"{row.get('id_documento', 'DOC')}.txt"),
        "titulo_documento": row.get("titulo_documento", "No determinado"),
        "tipo_norma": row.get("tipo_norma", "No determinado"),
        "numero_norma": row.get("numero_norma", "No determinado"),
        "entidad_emisora": row.get("entidad_emisora", "No determinado"),
        "fecha_publicacion": row.get("fecha_publicacion", "No determinado"),
        "tema_principal": row.get("tema_principal", "No determinado"),
        "subtema": row.get("subtema", "No determinado"),
        "fuente_oficial": row.get("fuente_oficial", "No determinado"),
        "url_fuente": row.get("url_fuente", "No determinado"),
        "estado_vigencia": row.get("estado_vigencia", "No determinado"),
        "prioridad": row.get("prioridad", "No determinado"),
        "tipo_chunk": tipo_chunk,
        "seccion": seccion,
        "titulo_seccion": titulo_seccion,
        "capitulo": capitulo,
        "anexo": anexo,
        "pagina_inicio": pagina_inicio,
        "pagina_fin": pagina_fin,
        "orden_chunk": orden_chunk,
        "suborden": suborden,
        "n_palabras": count_words(clean_chunk_text),
        "n_caracteres": len(clean_chunk_text),
        "texto": clean_chunk_text,
    }


def finalize_block(block, row, chunks, orden_chunk):
    """
    Finaliza un bloque actual y lo agrega a la lista de chunks.
    Si el bloque es muy grande, lo divide en subchunks.
    """
    if block is None:
        return orden_chunk

    text = normalize_text_for_chunk("\n".join(block["lines"]))
    word_count = count_words(text)

    if word_count < MIN_WORDS:
        return orden_chunk

    subtexts = split_large_text(text, MAX_WORDS, OVERLAP_WORDS)

    for suborder, subtext in enumerate(subtexts, start=1):
        chunk_record = create_chunk_record(
            row=row,
            text=subtext,
            tipo_chunk=block["tipo_chunk"],
            seccion=block["seccion"],
            titulo_seccion=block["titulo_seccion"],
            capitulo=block["capitulo"],
            anexo=block["anexo"],
            pagina_inicio=block["pagina_inicio"],
            pagina_fin=block["pagina_fin"],
            orden_chunk=orden_chunk,
            suborden=suborder
        )
        chunks.append(chunk_record)

    return orden_chunk + 1


def structural_chunk_document(text: str, row) -> list:
    """
    Genera chunks estructurales a partir del texto de un documento.
    El algoritmo recorre línea por línea y detecta páginas, títulos, capítulos,
    artículos, disposiciones y anexos.
    """
    chunks = []

    current_page = None
    current_title = "No determinado"
    current_chapter = "No determinado"
    current_annex = "No determinado"

    current_block = None
    orden_chunk = 1

    lines = text.splitlines()

    for raw_line in lines:
        line = normalize_line(raw_line)

        if not line:
            continue

        page_match = PAGE_MARKER_RE.match(line)
        if page_match:
            current_page = int(page_match.group(1))
            if current_block is not None:
                current_block["pagina_fin"] = current_page
            continue

        if TITLE_RE.match(line):
            current_title = line
            continue

        if CHAPTER_RE.match(line):
            current_chapter = line
            continue

        if ANNEX_RE.match(line):
            orden_chunk = finalize_block(current_block, row, chunks, orden_chunk)
            current_annex = line
            current_block = {
                "tipo_chunk": "anexo",
                "seccion": line,
                "titulo_seccion": current_title,
                "capitulo": current_chapter,
                "anexo": current_annex,
                "pagina_inicio": current_page,
                "pagina_fin": current_page,
                "lines": [line],
            }
            continue

        if DISPOSITION_RE.match(line):
            orden_chunk = finalize_block(current_block, row, chunks, orden_chunk)
            current_block = {
                "tipo_chunk": "disposicion",
                "seccion": line,
                "titulo_seccion": current_title,
                "capitulo": current_chapter,
                "anexo": current_annex,
                "pagina_inicio": current_page,
                "pagina_fin": current_page,
                "lines": [line],
            }
            continue

        if ARTICLE_RE.match(line):
            orden_chunk = finalize_block(current_block, row, chunks, orden_chunk)
            current_block = {
                "tipo_chunk": "articulo",
                "seccion": line,
                "titulo_seccion": current_title,
                "capitulo": current_chapter,
                "anexo": current_annex,
                "pagina_inicio": current_page,
                "pagina_fin": current_page,
                "lines": [line],
            }
            continue

        if LEGAL_HEADER_RE.match(line) and current_block is None:
            current_block = {
                "tipo_chunk": "encabezado_normativo",
                "seccion": line,
                "titulo_seccion": current_title,
                "capitulo": current_chapter,
                "anexo": current_annex,
                "pagina_inicio": current_page,
                "pagina_fin": current_page,
                "lines": [line],
            }
            continue

        if current_block is None:
            current_block = {
                "tipo_chunk": "preambulo",
                "seccion": "Preámbulo / considerandos",
                "titulo_seccion": current_title,
                "capitulo": current_chapter,
                "anexo": current_annex,
                "pagina_inicio": current_page,
                "pagina_fin": current_page,
                "lines": [],
            }

        current_block["lines"].append(line)
        current_block["pagina_fin"] = current_page

    orden_chunk = finalize_block(current_block, row, chunks, orden_chunk)

    return chunks


## 7. Prueba de chunking con el primer documento

Antes de procesar todo el corpus, se revisa un documento de ejemplo.


In [ ]:
first_row = df.iloc[0].to_dict()

archivo_txt = first_row.get("archivo_txt", f"{first_row['id_documento']}.txt")
txt_path = PROCESSED_DIR / archivo_txt

print(f"Documento de prueba: {first_row['id_documento']}")
print(f"Archivo TXT: {txt_path}")

if not txt_path.exists():
    print(f"No se encontró el archivo de texto: {txt_path}")
else:
    text = txt_path.read_text(encoding="utf-8", errors="ignore")
    sample_chunks = structural_chunk_document(text, first_row)

    print(f"Chunks generados para el documento de prueba: {len(sample_chunks)}")

    if sample_chunks:
        print("\nPrimer chunk generado:")
        print(json.dumps({k: v for k, v in sample_chunks[0].items() if k != "texto"}, ensure_ascii=False, indent=4))
        print("\nTexto del primer chunk:")
        print(sample_chunks[0]["texto"][:1500])


## 8. Generación de chunks para todos los documentos

Se procesan todos los textos extraídos y se genera un conjunto único de chunks.


In [ ]:
all_chunks = []
chunking_results = []

for _, row in df.iterrows():
    row_dict = row.to_dict()
    id_documento = str(row_dict["id_documento"])

    archivo_txt = row_dict.get("archivo_txt", None)

    if pd.isna(archivo_txt) or not archivo_txt:
        archivo_txt = f"{id_documento}.txt"

    txt_path = PROCESSED_DIR / str(archivo_txt)

    if not txt_path.exists():
        chunking_results.append({
            "id_documento": id_documento,
            "archivo_txt": str(archivo_txt),
            "estado_chunking": "TXT_NO_ENCONTRADO",
            "n_chunks": 0,
            "n_palabras_total_chunks": 0,
            "observacion": f"No existe {txt_path}",
        })
        continue

    text = txt_path.read_text(encoding="utf-8", errors="ignore")
    chunks = structural_chunk_document(text, row_dict)

    all_chunks.extend(chunks)

    chunking_results.append({
        "id_documento": id_documento,
        "archivo_txt": str(archivo_txt),
        "estado_chunking": "OK" if chunks else "SIN_CHUNKS",
        "n_chunks": len(chunks),
        "n_palabras_total_chunks": sum(chunk["n_palabras"] for chunk in chunks),
        "observacion": "Chunking completado" if chunks else "No se generaron chunks válidos",
    })

chunks_df = pd.DataFrame(all_chunks)
chunking_results_df = pd.DataFrame(chunking_results)

print(f"Total de chunks generados: {len(chunks_df)}")
chunks_df.head()


## 9. Resumen del chunking

In [ ]:
print("Resumen por estado de chunking:")
display(chunking_results_df["estado_chunking"].value_counts().reset_index().rename(columns={"index": "estado", "estado_chunking": "cantidad"}))

print("Resumen por tipo de chunk:")
if not chunks_df.empty:
    display(chunks_df["tipo_chunk"].value_counts().reset_index().rename(columns={"index": "tipo_chunk", "tipo_chunk": "cantidad"}))

print("Documentos con más chunks:")
display(chunking_results_df.sort_values("n_chunks", ascending=False).head(10))

print("Documentos con menos chunks:")
display(chunking_results_df.sort_values("n_chunks").head(10))


## 10. Control de calidad de chunks

Se revisan chunks demasiado pequeños o demasiado grandes.


In [ ]:
if chunks_df.empty:
    print("No hay chunks para revisar.")
else:
    small_chunks = chunks_df[chunks_df["n_palabras"] < MIN_WORDS]
    large_chunks = chunks_df[chunks_df["n_palabras"] > MAX_WORDS]

    print(f"Chunks pequeños (< {MIN_WORDS} palabras): {len(small_chunks)}")
    print(f"Chunks grandes (> {MAX_WORDS} palabras): {len(large_chunks)}")

    print("\nDistribución de palabras por chunk:")
    display(chunks_df["n_palabras"].describe())

    if not large_chunks.empty:
        print("\nMuestra de chunks grandes:")
        display(large_chunks[["id_chunk", "id_documento", "tipo_chunk", "seccion", "n_palabras"]].head(10))


## 11. Muestra aleatoria de chunks

In [ ]:
if not chunks_df.empty:
    sample_display = chunks_df.sample(min(5, len(chunks_df)), random_state=42)

    for _, chunk in sample_display.iterrows():
        print("=" * 100)
        print(f"ID chunk: {chunk['id_chunk']}")
        print(f"Documento: {chunk['id_documento']} | {chunk['titulo_documento']}")
        print(f"Tipo chunk: {chunk['tipo_chunk']}")
        print(f"Sección: {chunk['seccion']}")
        print(f"Páginas: {chunk['pagina_inicio']} - {chunk['pagina_fin']}")
        print(f"Palabras: {chunk['n_palabras']}")
        print("-" * 100)
        print(chunk["texto"][:1000])
        print()


## 12. Guardado de chunks y reportes

Se guardan los chunks en dos formatos:

- `JSONL`: recomendado para procesamiento posterior.
- `CSV`: útil para revisión en Excel o Google Sheets.


In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

chunks_jsonl_path = CHUNKS_DIR / "chunks_normativa_v1.jsonl"
chunks_csv_path = CHUNKS_DIR / "chunks_normativa_v1.csv"
chunking_report_path = REPORTS_DIR / f"reporte_chunking_{timestamp}.csv"
chunking_summary_path = REPORTS_DIR / f"resumen_chunking_{timestamp}.json"

if not chunks_df.empty:
    with open(chunks_jsonl_path, "w", encoding="utf-8") as f:
        for chunk in all_chunks:
            f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

    chunks_df.to_csv(chunks_csv_path, index=False, encoding="utf-8-sig")

chunking_results_df.to_csv(chunking_report_path, index=False, encoding="utf-8-sig")

summary = {
    "fecha_chunking": datetime.now().isoformat(timespec="seconds"),
    "total_documentos": int(len(df)),
    "total_chunks": int(len(chunks_df)),
    "max_words": MAX_WORDS,
    "overlap_words": OVERLAP_WORDS,
    "min_words": MIN_WORDS,
    "documentos_ok": int((chunking_results_df["estado_chunking"] == "OK").sum()),
    "documentos_sin_chunks": int((chunking_results_df["estado_chunking"] == "SIN_CHUNKS").sum()),
    "documentos_txt_no_encontrado": int((chunking_results_df["estado_chunking"] == "TXT_NO_ENCONTRADO").sum()),
}

if not chunks_df.empty:
    summary["promedio_palabras_por_chunk"] = float(chunks_df["n_palabras"].mean())
    summary["min_palabras_por_chunk"] = int(chunks_df["n_palabras"].min())
    summary["max_palabras_por_chunk"] = int(chunks_df["n_palabras"].max())
else:
    summary["promedio_palabras_por_chunk"] = 0
    summary["min_palabras_por_chunk"] = 0
    summary["max_palabras_por_chunk"] = 0

with open(chunking_summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=4)

print(f"Chunks JSONL guardados en: {chunks_jsonl_path}")
print(f"Chunks CSV guardados en: {chunks_csv_path}")
print(f"Reporte de chunking guardado en: {chunking_report_path}")
print(f"Resumen JSON guardado en: {chunking_summary_path}")


## 13. Resultado final

Si el chunking se generó correctamente, el corpus queda listo para la siguiente fase:

**Notebook 04 - Embeddings e indexación vectorial**.


In [ ]:
if chunks_df.empty:
    print("No se generaron chunks. Revisa los textos extraídos o las reglas de segmentación.")
else:
    print("Chunking completado correctamente.")
    print(f"Total de chunks generados: {len(chunks_df)}")
    print("Siguiente paso: generar embeddings e indexar en la base vectorial.")

display(chunking_results_df)
